In [62]:
#!pip install sympy

In [63]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from qhdopt import QHD

In [64]:
# 初始化（用默认 3-bus 数据）
model = SympyACOPFModel()   # 打开 proximal 选项，后面会用到

In [65]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()

rho = 10.0
alpha = 5e-1   # 对偶步长尽量小一点
max_outer = 50
tol = 1e-4

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    print(f"\n--- Outer Iteration {k} ---")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=1500.0
        )

    option = 1
    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)

        qhd_model.simbi_setup(
            resolution=6,
            agents=1024,
            max_steps=1000,
            embedding_scheme="unary",
            post_processing_method='TNC',
            verbose=True
        )

        response = qhd_model.optimize(verbose=0)

        x_new = np.asarray(response.refined_minimizer)

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    PrintQHDACOPFResults(model, x_new)

    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=rho,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    rho_max = 1e4
    if np.linalg.norm(h_x) > 0.5 * np.linalg.norm(h_old) and rho < rho_max:
        rho *= 2
        print("Increasing rho to", rho)

    # ======================================
    # (7) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(xk)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (6) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")


===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---


Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.6050112247467041
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	0.8833	0.0139	0.1319	-0.1353	1.0000	0.0000	0.0000	0.0000
2	0.9048	0.0101	0.0000	0.0861	1.0000	0.0000	0.0000	0.0000
3	0.9219	-0.0160	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.1319	-0.0492		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.0002	0.0454	-0.0260	0.3240
1	3	-0.0473	-0.0922	-0.0004	0.2172
2	3	-0.0372	-0.0961	0.0000	0.1242


Total Ploss: -0.2272
Total Qloss: 0.6390
Total Load Supplied: 119.6855%
||h(x)|| = 0.8587160424964875
[rho-check] ||h_old||=4.798e-01, ||h_new||=8.587e-01, rho=10
Increasing rho to 20.0
Constraint check: False

--- Outer Iteration 1 ---


Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.590468168258667
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	0.9135	0.0296	0.0237	-0.1578	1.0000	0.0000	0.0000	0.0000
2	0.9086	0.0199	0.0000	0.0816	1.0000	0.0000	0.0000	0.0000
3	0.9482	0.0111	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.0237	-0.0762		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	-0.0222	-0.1613	-0.0619	-0.0274
1	3	-0.0874	-0.0507	0.0419	0.1890
2	3	0.0259	0.0185	0.0116	0.2098


Total Ploss: -0.2773
Total Qloss: 0.3630
Total Load Supplied: 100.3577%
||h(x)|| = 0.848360420926248
[rho-check] ||h_old||=8.587e-01, ||h_new||=8.484e-01, rho=20
Increasing rho to 40.0
Constraint check: False

--- Outer Iteration 2 ---


...

Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.4685084819793701
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	0.9278	0.4633	0.2137	-0.0691	1.0000	0.0000	0.0000	0.0000
2	0.9321	0.4624	0.1553	0.1104	1.0000	0.0000	0.0000	0.0000
3	0.9288	0.4342	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.3690	0.0413		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.0741	-0.0256	-0.0981	0.0705
1	3	0.2138	-0.1548	0.0327	-0.0254
2	3	0.1608	-0.1551	0.0679	-0.0371


Total Ploss: 0.1133
Total Qloss: 0.0105
Total Load Supplied: 85.2344%
||h(x)|| = 0.48113418604154284
[rho-check] ||h_old||=8.105e-01, ||h_new||=4.811e-01, rho=1.28e+03
Increasing rho to 2560.0
Constraint check: False

--- Outer Iteration 10 ---


Bifurcated agents:   0%|          | 0/1024 [00:00<?, ?it/s]


backend time consumption: 0.4564950466156006


KeyboardInterrupt: 

In [ ]:
response.refined_minimizer


array([ 0.06287919,  0.        , -0.18404029,  0.06542087,  0.84820733,
        0.87069852,  0.86101576,  0.48281526,  0.50053502,  0.46616191,
        1.11879722,  1.12069973,  1.11773689, -0.03452672, -0.11160269,
       -0.01119233, -0.02417105,  0.02962917, -0.01453296,  0.        ,
        0.0142531 ,  0.        ])

In [69]:
qhd_model.func

0.001875*P_G0**2 + 3.4425948363635*P_G0 + 0.00875*P_G1**2 + 0.0263187779498779*P_G1 + 18.368816888402*P_ij0 - 13.8976825906162*P_ij1 - 61.9848688383167*P_ij2 - 134.358944132099*Q_G0 + 2.64528298745575*Q_G1 - 14.2995695179608*Q_ij0 + 46.8267619108008*Q_ij1 - 6.70085160088896*Q_ij2 - 26.3772785094347*S_tot_sq0 - 26.8401375622945*S_tot_sq1 - 29.1390415913141*S_tot_sq2 + 1592.71073312955*V_I0 - 180.867850245576*V_I1 + 295.228433626218*V_I2 + 2643.62484690661*V_R0 - 2848.68509963142*V_R1 - 110.824908639837*V_R2 + 38.5316367139767*V_sq0 + 24.2322381262548*V_sq1 + 91.4846225204114*V_sq2 + 750.0*(P_G0 - 0.213698726645926)**2 + 750.0*(P_G1 - 0.155276564381654)**2 + 750.0*(P_ij0 - 0.0741468676762052)**2 + 750.0*(P_ij1 - 0.213782115574128)**2 + 750.0*(P_ij2 - 0.16075253267353)**2 + 750.0*(Q_G0 + 0.0690502589149169)**2 + 750.0*(Q_G1 - 0.110383567869368)**2 + 750.0*(Q_ij0 + 0.0981439728961718)**2 + 750.0*(Q_ij1 - 0.0327003570074135)**2 + 750.0*(Q_ij2 - 0.0678652294312309)**2 + 750.0*(S_tot_sq0 - 0.

In [ ]:
print(model.P_D, model.Q_D, model.P_G, model.Q_G)
print(model.variable_list)
print(model.Var_bound_list)

[0.  0.  0.3] [0.  0.  0.1] (P_G0, P_G1) (Q_G0, Q_G1)
[P_G0, P_G1, Q_G0, Q_G1, V_R0, V_R1, V_R2, V_I0, V_I1, V_I2, V_sq0, V_sq1, V_sq2, P_ij0, P_ij1, P_ij2, Q_ij0, Q_ij1, Q_ij2, S_tot_sq0, S_tot_sq1, S_tot_sq2]
[[0.0, 2.0], [0.0, 2.0], [-2.0, 10.0], [-2.0, 10.0], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [0.81, 1.2100000000000002], [0.81, 1.2100000000000002], [0.81, 1.2100000000000002], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [0.0, 9.0], [0.0, 9.0], [0.0, 9.0]]
